<a href="https://colab.research.google.com/github/HoaiNam2k5/IOT_LAB04_NguyenHoaiNam/blob/main/LAB05_NguyenHoaiNam_2001230530.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BÁO CÁO BÀI TẬP: XÂY DỰNG HỆ THỐNG CẢNH BÁO IOT QUA GIAO THỨC MQTT

---

## 1. Khởi Tạo Môi Trường Hệ Thống
Tiến hành cài đặt thư viện `paho-mqtt` để hỗ trợ thiết lập kết nối Client-Broker theo tiêu chuẩn giao thức MQTT.

In [11]:
# Cài đặt thư viện Eclipse Paho MQTT phiên bản mới nhất
!pip install paho-mqtt

## 2. Thiết Lập Trạm Giám Sát Trung Tâm (MQTT Subscriber)
Trạm giám sát đóng vai trò nhận dữ liệu thời gian thực từ cảm biến, phân tích cú pháp dữ liệu JSON và đưa ra các cảnh báo tự động dựa trên ngưỡng cấu hình sẵn:
* **Nhiệt độ > 45°C**: Đưa ra cảnh báo Nhiệt độ quá cao (NÓNG) ⚠️
* **Độ ẩm < 50%**: Đưa ra cảnh báo Độ ẩm quá thấp (KHÔ) ⚠️

In [12]:
import paho.mqtt.client as mqtt
import json
import time

# =====================================================================
# 2.1 CẤU HÌNH THÔNG SỐ KẾT NỐI BROKER
# =====================================================================
MQTT_BROKER = "broker.hivemq.com"   # Sử dụng Broker công cộng HiveMQ
MQTT_PORT = 1883                    # Cổng kết nối TCP tiêu chuẩn không mã hóa
KEEP_ALIVE_INTERVAL = 60            # Chu kỳ gửi gói tin giữ kết nối (giây)
MONITOR_TOPIC = "iot/test/topic"    # Chủ đề giám sát dữ liệu cảm biến

# =====================================================================
# 2.2 ĐỊNH NGHĨA CÁC HÀM XỬ LÝ SỰ KIỆN (CALLBACK FUNCTIONS)
# =====================================================================

def on_connect_handler(client, userdata, flags, reason_code, properties=None):
    """Hàm tự động gọi khi Client kết nối thành công hoặc thất bại với Broker"""
    if reason_code == 0:
        print("\n[HỆ THỐNG] Kết nối tới Broker thành công!")
        client.subscribe(MONITOR_TOPIC)
        print(f"[HỆ THỐNG] Đang lắng nghe luồng dữ liệu tại Topic: '{MONITOR_TOPIC}'")
    else:
        print(f"[HỆ THỐNG] Lỗi kết nối! Mã phản hồi từ Broker: {reason_code}")

def on_message_handler(client, userdata, msg):
    """Hàm tự động xử lý và phân tích dữ liệu khi có tin nhắn mới gửi đến"""
    try:
        # Giải mã mảng bytes nhận được từ Payload sang chuỗi ký tự UTF-8
        payload_str = msg.payload.decode("utf-8")
        print(f"\n──────────────────────────────────────────────────")
        print(f"📬 [TIN NHẮN ĐẾN] Topic: {msg.topic}")
        print(f"📦 [DỮ LIỆU THÔ] Payload: {payload_str}")

        # Phân tích cú pháp dữ liệu dạng chuỗi JSON
        sensor_payload = json.loads(payload_str)
        temp_val = sensor_payload.get("temperature")
        humid_val = sensor_payload.get("humidity")

        # --- THỰC HIỆN KIỂM TRA ĐIỀU KIỆN CẢNH BÁO ---

        # 1. Logic kiểm tra nhiệt độ
        if temp_val is not None:
            if temp_val > 45:
                print(f"🔥 CẢNH BÁO: Nhiệt độ đo được là {temp_val}°C -> VƯỢT NGƯỠNG NGUY HIỂM (>45°C)!")
            else:
                print(f"✅ An toàn: Nhiệt độ đo được là {temp_val}°C (Trong ngưỡng an toàn)")

        # 2. Logic kiểm tra độ ẩm
        if humid_val is not None:
            if humid_val < 50:
                print(f"⚠️ CẢNH BÁO: Độ ẩm đo được là {humid_val}% -> QUÁ KHÔ HẠN (<50%)!")
            else:
                print(f"✅ An toàn: Độ ẩm ổn định ở mức {humid_val}%")

    except json.JSONDecodeError:
        print("❌ Lỗi định dạng: Dữ liệu nhận được không phải cấu trúc JSON hợp lệ!")
    except Exception as e:
        print(f"❌ Đã xảy ra lỗi không xác định trong quá trình phân tích: {e}")

# =====================================================================
# 2.3 KHỞI TẠO VÀ CHẠY TIẾN TRÌNH SUBSCRIBER
# =====================================================================
sub_client_id = f"iot-sub-station-{int(time.time())}"
subscriber_instance = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, sub_client_id)

# Gán các hàm callback đã định nghĩa vào Client Instance
subscriber_instance.on_connect = on_connect_handler
subscriber_instance.on_message = on_message_handler

# Kết nối đến Broker và khởi chạy tiến trình nhận tin ngầm
print(f"[HỆ THỐNG] Tiến hành kết nối tới Broker {MQTT_BROKER}...")
subscriber_instance.connect(MQTT_BROKER, MQTT_PORT, KEEP_ALIVE_INTERVAL)
subscriber_instance.loop_start()
print("[HỆ THỐNG] Tiến trình lắng nghe dữ liệu đã sẵn sàng!")

[HỆ THỐNG] Tiến hành kết nối tới Broker broker.hivemq.com...
[HỆ THỐNG] Tiến trình lắng nghe dữ liệu đã sẵn sàng!


## 3. Mô Phỏng Thiết Bị Đầu Cuối (MQTT Publisher)
Dưới đây là chương trình mô phỏng một cảm biến IoT tại thực địa gửi định kỳ các thông số về Nhiệt độ & Độ ẩm dưới định dạng gói tin JSON.

In [13]:
import paho.mqtt.client as mqtt
import json
import time

# Cấu hình đích đến cho Publisher
BROKER_ADDRESS = "broker.hivemq.com"
TARGET_TOPIC = "iot/test/topic"

# Khởi tạo một Publisher Client tạm thời
pub_client_id = f"iot-pub-sensor-{int(time.time())}"
publisher_instance = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, pub_client_id)

# Tiến hành kết nối
publisher_instance.connect(BROKER_ADDRESS, 1883, 60)

# Tạo lập dữ liệu giả lập (đáp ứng điều kiện cảnh báo)
mock_data = {
    "temperature": 48,  # Nhiệt độ cao (> 45)
    "humidity": 42      # Độ ẩm thấp (< 50)
}

# Chuyển đổi sang định dạng JSON
payload_json = json.dumps(mock_data)
print(f"[PUBLISHER] Đang phát dữ liệu cảm biến: {payload_json} lên Topic: '{TARGET_TOPIC}'...")

# Thực hiện xuất bản gói tin
publisher_instance.publish(TARGET_TOPIC, payload_json)

# Giải phóng kết nối an toàn
publisher_instance.disconnect()
print("[PUBLISHER] Đã ngắt kết nối an toàn sau khi gửi dữ liệu thành công!")

[PUBLISHER] Đang phát dữ liệu cảm biến: {"temperature": 48, "humidity": 42} lên Topic: 'iot/test/topic'...
[PUBLISHER] Đã ngắt kết nối an toàn sau khi gửi dữ liệu thành công!
